# Context Managers

The `with` statement ensures cleanup happens, even if errors occur. It's everywhere in AI code.

## The Problem: Resource Management

In [ ]:
# Without context manager - manual cleanup
# (Don't do this!)

# file = open("data.txt", "w")
# try:
#     file.write("Hello")
#     # What if error happens here?
# finally:
#     file.close()  # Must remember to close!

print("Manual cleanup is error-prone")

In [ ]:
# With context manager - automatic cleanup
# (Do this!)

# with open("data.txt", "w") as file:
#     file.write("Hello")
# # File automatically closed, even if error!

print("Context managers handle cleanup automatically")

## How `with` Works

```python
with expression as variable:
    # use variable
# cleanup happens automatically
```

Internally:
1. `__enter__()` is called → returns value for `as`
2. Your code runs
3. `__exit__()` is called → even if exception

In [ ]:
# Simple demonstration
class DemoContext:
    def __enter__(self):
        print("1. Entering context")
        return "resource"  # This becomes 'as' variable
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        print("3. Exiting context (cleanup)")
        return False  # Don't suppress exceptions

with DemoContext() as resource:
    print(f"2. Using: {resource}")

print("4. After context")

## Common Built-in Context Managers

In [ ]:
# File handling (most common)
# with open("file.txt") as f:
#     content = f.read()

# Threading locks
import threading
lock = threading.Lock()

with lock:
    print("Lock acquired, doing work...")
# Lock released automatically

print("Lock released")

In [ ]:
# Suppress exceptions
from contextlib import suppress

with suppress(FileNotFoundError):
    # This won't crash if file doesn't exist
    # open("nonexistent.txt").read()
    pass

print("Continued after suppressed error")

In [ ]:
# Temporary directory
import tempfile
import os

with tempfile.TemporaryDirectory() as tmpdir:
    print(f"Temp directory: {tmpdir}")
    print(f"Exists: {os.path.exists(tmpdir)}")

print(f"After context - exists: {os.path.exists(tmpdir)}")

## Multiple Context Managers

In [ ]:
# Multiple managers - comma separated
# with open("input.txt") as infile, open("output.txt", "w") as outfile:
#     content = infile.read()
#     outfile.write(content.upper())

# Or with parentheses (Python 3.10+)
# with (
#     open("input.txt") as infile,
#     open("output.txt", "w") as outfile,
# ):
#     pass

print("Both files handled properly")

## Creating Context Managers

In [ ]:
# Method 1: Class with __enter__ and __exit__
import time

class Timer:
    def __enter__(self):
        self.start = time.time()
        return self
    
    def __exit__(self, *args):
        self.elapsed = time.time() - self.start
        print(f"Elapsed: {self.elapsed:.4f}s")
        return False

with Timer():
    # Some work
    total = sum(range(1000000))
    print(f"Sum: {total}")

In [ ]:
# Method 2: @contextmanager decorator (easier!)
from contextlib import contextmanager

@contextmanager
def timer():
    start = time.time()
    yield  # Everything before yield is __enter__
    # Everything after yield is __exit__
    elapsed = time.time() - start
    print(f"Elapsed: {elapsed:.4f}s")

with timer():
    total = sum(range(1000000))
    print(f"Sum: {total}")

In [ ]:
# Context manager that yields a value
@contextmanager
def managed_resource(name):
    print(f"Setting up {name}")
    resource = {"name": name, "connected": True}
    try:
        yield resource  # This becomes the 'as' variable
    finally:
        print(f"Cleaning up {name}")
        resource["connected"] = False

with managed_resource("database") as db:
    print(f"Using: {db}")
    # Work with db...

## AI Code Patterns

In [ ]:
# Pattern 1: Database connections
@contextmanager
def get_db_connection():
    print("Opening database connection")
    connection = {"status": "connected"}  # Simulated
    try:
        yield connection
    finally:
        print("Closing database connection")
        connection["status"] = "closed"

with get_db_connection() as conn:
    print(f"Running query on {conn}")

In [ ]:
# Pattern 2: Temporary state changes
import os

@contextmanager
def temporary_env(key, value):
    """Temporarily set an environment variable."""
    old_value = os.environ.get(key)
    os.environ[key] = value
    try:
        yield
    finally:
        if old_value is None:
            del os.environ[key]
        else:
            os.environ[key] = old_value

print(f"Before: DEBUG = {os.environ.get('DEBUG')}")

with temporary_env("DEBUG", "true"):
    print(f"Inside: DEBUG = {os.environ.get('DEBUG')}")

print(f"After: DEBUG = {os.environ.get('DEBUG')}")

In [ ]:
# Pattern 3: Change directory temporarily
@contextmanager
def working_directory(path):
    """Temporarily change working directory."""
    old_dir = os.getcwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(old_dir)

print(f"Current: {os.getcwd()}")
# with working_directory("/tmp"):
#     print(f"Inside: {os.getcwd()}")
# print(f"After: {os.getcwd()}")

In [ ]:
# Pattern 4: Timing sections of code
@contextmanager
def timed_section(name):
    start = time.time()
    print(f"Starting: {name}")
    try:
        yield
    finally:
        elapsed = time.time() - start
        print(f"Finished: {name} ({elapsed:.4f}s)")

with timed_section("data processing"):
    data = list(range(100000))
    result = sum(x**2 for x in data)

## `contextlib` Utilities

In [ ]:
from contextlib import redirect_stdout, redirect_stderr
import io

# Capture stdout to string
output = io.StringIO()
with redirect_stdout(output):
    print("This goes to the string")
    print("So does this")

captured = output.getvalue()
print(f"Captured: {repr(captured)}")

In [ ]:
from contextlib import ExitStack

# Manage multiple context managers dynamically
# with ExitStack() as stack:
#     files = [stack.enter_context(open(f)) for f in filenames]
#     # All files will be closed when exiting

print("ExitStack manages multiple contexts")

## Summary

| Pattern | Purpose |
|---------|--------|
| `with open() as f:` | File handling |
| `with lock:` | Thread synchronization |
| `with suppress(Error):` | Ignore specific errors |
| `@contextmanager` | Create custom managers |
| `yield` in context | Value available as `as` |
| `finally` block | Guaranteed cleanup |

## Module Complete!

You now understand:
- Conditionals and truthiness
- Loops and comprehensions
- Exception handling
- Context managers and `with`

Next module: Data Structures!